In [ ]:
import math 
import numpy as np
from tqdm import tqdm
import timeit
import matplotlib.pyplot as plt
from quantum_close_neighbors.grovers import (
    Grovers,
    ProbabilisticGrovers, 
    QiskitGroversPhaseOracle,
    QiskitGroversStaticPhaseOracle)
from quantum_close_neighbors.combarro import algorithm_1, algorithm_2, algorithm_3
from quantum_close_neighbors.close_neighbors import (close_neighbors, 
                             close_neighbors_2,
                             CloseNeighborsHyperparameters, 
                             CloseNeighborsResult, 
                             DEFAULT_HYPERPARAMETERS)

### Plotting

In [ ]:

def plot_runtimes(runtimes, bins, alogrithm_name):
    plt.hist(runtimes, bins=bins)
    plt.xlabel("Runtime (s)")
    plt.ylabel("Frequency")
    plt.title("Runtime Distribution for " + alogrithm_name)
    plt.show()

def plot_grover_iterations(grover_iterations, bins, algorithm_name):
    plt.hist(grover_iterations, bins=bins)
    plt.xlabel("Number of Grover Iterations")
    plt.ylabel("Frequency")
    plt.title("Number of Grover Iterations for " + algorithm_name)
    plt.show()

def plot_proportion_of_marked_elements(proportion_of_marked_elements_found, algorithm_name):
    plt.hist(proportion_of_marked_elements_found, bins=20)
    plt.xlabel("Proportion of Marked Elements Found")
    plt.ylabel("Frequency")
    plt.title("Proportion of Marked Elements Found for " + algorithm_name)
    plt.show()

### Dataset Construction

In [ ]:
n_particles = 1000
n_neighbors = 30

nu = math.comb(n_particles, 2)          # nu is n choose 2
mu = n_particles * n_neighbors // 2     # mu is the number of close neighbors

# Number of runs for each algorithm
RUNS_CLOSE_NEIGHBORS = 5
RUNS_CLOSE_NEIGHBORS_V2 = 5
RUNS_ALG1 = 5
RUNS_ALG2 = 5
RUNS_ALG3 = 5

# RUNTIME BINS
RUNTIME_BINS = 10
GROVER_ITERATION_BINS = 10

# Desired rate of success
# Success rate is defined as the proportion of runs that found all marked elements
DESIRED_SUCCESS_RATE = 0.99

In [ ]:
# Build the grovers object
GROVER_IMPLEMENTATION = "probabilistic"

def build_grovers(n_particles, n_neighbors, with_replacement = False) -> Grovers:
    
    if GROVER_IMPLEMENTATION == "probabilistic":
        grovers = ProbabilisticGrovers(n_particles, n_neighbors)
    elif GROVER_IMPLEMENTATION == "qiskit":
        if with_replacement:
            grovers = QiskitGroversStaticPhaseOracle.init_with_random_marked_elements(n_particles, n_neighbors)
        else:
            grovers = QiskitGroversPhaseOracle.init_with_random_marked_elements(n_particles, n_neighbors)

    else:
        raise ValueError(f"Invalid Grover implementation: {GROVER_IMPLEMENTATION}")

    return grovers

### Close Neighbors with Replacement

In [ ]:
# DESIRED_SUCCESS_RATE = 1 - n^(-alpha)
# n^(-alpha) = 1 - DESIRED_SUCCESS_RATE
# -alpha * log(n) = log(1 - DESIRED_SUCCESS_RATE)
# alpha = log(1 - DESIRED_SUCCESS_RATE) / log(n)
alpha = -math.log(1 - DESIRED_SUCCESS_RATE) / math.log(nu)

successes = 0
proportion_of_marked_elements_found = []
runtimes = []
grover_iterations_precritical = []
grover_iterations_postcritical = []
grover_iterations_finalcollection = []
grover_iterations = []
grover_measurements = []

for _ in tqdm(range(RUNS_CLOSE_NEIGHBORS)):

    start = timeit.default_timer()
    result = close_neighbors(nu, alpha, hyperparameters=DEFAULT_HYPERPARAMETERS, grover=build_grovers(nu, mu, with_replacement=True))
    end = timeit.default_timer()

    precritical, postcritical, final_collection = result.iterations
    runtimes.append(end - start)
    grover_iterations_precritical.append(precritical)
    grover_iterations_postcritical.append(postcritical)
    grover_iterations_finalcollection.append(final_collection)
    grover_iterations.append(precritical + postcritical + final_collection)
    grover_measurements.append(result.grover_measurements)

    if result.n_close_neighbors == mu:
        successes += 1
    proportion_of_marked_elements_found.append(result.n_close_neighbors / mu)

success_rate = successes / RUNS_CLOSE_NEIGHBORS if RUNS_CLOSE_NEIGHBORS > 0 else 0

if RUNS_CLOSE_NEIGHBORS > 0:
    print(f"Our Algorithm, N={nu}, alpha={alpha}, runs={RUNS_CLOSE_NEIGHBORS}")
    print(f"Success rate: {success_rate}")
    print(f"Average runtime: {np.mean(runtimes)} seconds")
    print(f"Average number of Grover iterations: {np.mean(grover_iterations)}")
    print(f"Average number of Grover iterations precritical: {np.mean(grover_iterations_precritical)}")
    print(f"Average number of Grover iterations postcritical: {np.mean(grover_iterations_postcritical)}")
    print(f"Average number of Grover iterations final collection: {np.mean(grover_iterations_finalcollection)}")
    print(f"Average number of Grover measurements: {np.mean(grover_measurements)}")
    print(f"Average proportion of marked elements found: {np.mean(proportion_of_marked_elements_found)}")

    #plot_runtimes(runtimes, RUNTIME_BINS, f"Our Algorithm")
    plot_grover_iterations(grover_iterations, GROVER_ITERATION_BINS, f"Our Algorithm" )
    plot_proportion_of_marked_elements(proportion_of_marked_elements_found, f"Our Algorithm")

### Close Neighbors without Replacement

In [ ]:
# DESIRED_SUCCESS_RATE = 1 - n^(-alpha)
# n^(-alpha) = 1 - DESIRED_SUCCESS_RATE
# -alpha * log(n) = log(1 - DESIRED_SUCCESS_RATE)
# alpha = log(1 - DESIRED_SUCCESS_RATE) / log(n)


# TODO: Modifying hyperparams to see if we can get better results
UPDATED_HYPERPARAMETERS = CloseNeighborsHyperparameters(
    final_collection_k=lambda n, alpha, A1 : (alpha+1) * 4 * A1 * np.log(n),
    m_factor_increase=1,
    c1=450,
    c2=6,
    lambda_val=6/5
)


alpha = -math.log(1 - DESIRED_SUCCESS_RATE) / math.log(nu)

successes = 0
proportion_of_marked_elements_found = []
runtimes = []
grover_iterations_precritical = []
grover_iterations_postcritical = []
grover_iterations_finalcollection = []
grover_iterations = []
grover_measurements = []

for _ in tqdm(range(RUNS_CLOSE_NEIGHBORS_V2)):

    start = timeit.default_timer()
    result : CloseNeighborsResult = close_neighbors_2(
        nu, 
        alpha, 
        hyperparameters=UPDATED_HYPERPARAMETERS, 
        grover=build_grovers(nu, mu),
        verbose=False)
    end = timeit.default_timer()

    precritical, postcritical, final_collection = result.iterations
    runtimes.append(end - start)
    grover_iterations_precritical.append(precritical)
    grover_iterations_postcritical.append(postcritical)
    grover_iterations_finalcollection.append(final_collection)
    grover_iterations.append(precritical + postcritical + final_collection)
    grover_measurements.append(result.grover_measurements)

    if result.n_close_neighbors == mu:
        successes += 1
    proportion_of_marked_elements_found.append(result.n_close_neighbors / mu)

success_rate = successes / RUNS_CLOSE_NEIGHBORS_V2 if RUNS_CLOSE_NEIGHBORS_V2 > 0 else 0

print(f"Our Algorithm, N={nu}, alpha={alpha}, runs={RUNS_CLOSE_NEIGHBORS_V2}")
print(f"Success rate: {success_rate}")
print(f"Average runtime: {np.mean(runtimes)} seconds")
print(f"Average number of Grover iterations: {np.mean(grover_iterations)}")
print(f"Average number of Grover iterations precritical: {np.mean(grover_iterations_precritical)}")
print(f"Average number of Grover iterations postcritical: {np.mean(grover_iterations_postcritical)}")
print(f"Average number of Grover iterations final collection: {np.mean(grover_iterations_finalcollection)}")
print(f"Average number of Grover measurements: {np.mean(grover_measurements)}")
print(f"Average proportion of marked elements found: {np.mean(proportion_of_marked_elements_found)}")

#plot_runtimes(runtimes, RUNTIME_BINS, f"Our Algorithm")
plot_grover_iterations(grover_iterations, GROVER_ITERATION_BINS, f"Our Algorithm" )
plot_proportion_of_marked_elements(proportion_of_marked_elements_found, f"Our Algorithm")


### Algorithm 1

In [ ]:
error_bound = 1 - DESIRED_SUCCESS_RATE # omega 

successes = 0
proportion_of_marked_elements_found = []
runtimes = []
grover_iterations = []
grover_measurements = []

for _ in tqdm(range(RUNS_ALG1)):

    start = timeit.default_timer()
    marked_elements, iterations, measurements = algorithm_1(nu, mu, error_bound, grover=build_grovers(nu, mu, with_replacement=True))
    end = timeit.default_timer()

    runtimes.append(end - start)
    grover_iterations.append(iterations)
    grover_measurements.append(measurements)
    if marked_elements == mu:
        successes += 1
    proportion_of_marked_elements_found.append(marked_elements / mu)

success_rate = successes / RUNS_ALG1 if RUNS_ALG1 > 0 else 0

if RUNS_ALG1 > 0:

    print(f"Algorithm 1, nu={nu}, mu={mu}, error_bound={error_bound}, runs={RUNS_ALG1}")
    print(f"Success rate: {success_rate}")
    print(f"Average runtime: {np.mean(runtimes)} seconds")
    print(f"Average number of Grover iterations: {np.mean(grover_iterations)}")
    print(f"Average proportion of marked elements found: {np.mean(proportion_of_marked_elements_found)}")
    print(f"Average number of Grover measurements: {np.mean(grover_measurements)}")


    plot_runtimes(runtimes, RUNTIME_BINS, "Algorithm 1")
    plot_grover_iterations(grover_iterations, GROVER_ITERATION_BINS, "Algorithm 1")
    plot_proportion_of_marked_elements(proportion_of_marked_elements_found, "Algorithm 1")

### Algorithm 2

In [ ]:
error_bound = 1 - DESIRED_SUCCESS_RATE # omega 
B = int(3*nu/4) # Upper bound on the number of marked elements (<= mu <= 3*nu/4)

successes = 0
proportion_of_marked_elements_found = []
runtimes = []
grover_iterations = []
grover_measurements = []


for _ in tqdm(range(RUNS_ALG2)):

    start = timeit.default_timer()
    marked_elements, iterations, measurements = algorithm_2(nu, B, error_bound, grover=build_grovers(nu, mu))
    end = timeit.default_timer()

    runtimes.append(end - start)
    grover_iterations.append(iterations)
    grover_measurements.append(measurements)
    if marked_elements == mu:
        successes += 1
    proportion_of_marked_elements_found.append(marked_elements / mu)

success_rate = successes / RUNS_ALG2 if RUNS_ALG2 > 0 else 0


if RUNS_ALG2 > 0:

    print(f"Algorithm 2, nu={nu}, mu={mu}, error_bound={error_bound}, runs={RUNS_ALG2}")
    print(f"Success rate: {success_rate}")
    print(f"Average runtime: {np.mean(runtimes)} seconds")
    print(f"Average number of Grover iterations: {np.mean(grover_iterations)}")
    print(f"Average proportion of marked elements found: {np.mean(proportion_of_marked_elements_found)}")
    print(f"Average number of Grover measurements: {np.mean(grover_measurements)}")

    plot_runtimes(runtimes, RUNTIME_BINS, "Algorithm 2")
    plot_grover_iterations(grover_iterations, GROVER_ITERATION_BINS, "Algorithm 2")
    plot_proportion_of_marked_elements(proportion_of_marked_elements_found, "Algorithm 2")

### Algorithm 3

In [ ]:
error_bound = 1 - DESIRED_SUCCESS_RATE # omega 
B = int(3*nu/4) # Upper bound on the number of marked elements (<= mu <= 3*nu/4)

successes = 0
proportion_of_marked_elements_found = []
runtimes = []
grover_iterations = []
grover_measurements = []

for _ in tqdm(range(RUNS_ALG3)):

    start = timeit.default_timer()
    marked_elements, iterations, measurements = algorithm_3(nu, B, error_bound, grover=build_grovers(nu, mu))
    end = timeit.default_timer()

    runtimes.append(end - start)
    grover_iterations.append(iterations)
    grover_measurements.append(measurements)
    if marked_elements == mu:
        successes += 1
    proportion_of_marked_elements_found.append(marked_elements / mu)

success_rate = successes / RUNS_ALG3 if RUNS_ALG3 > 0 else 0

if RUNS_ALG3 > 0:
    print(f"Algorithm 3, nu={nu}, mu={mu}, error_bound={error_bound}, runs={RUNS_ALG3}")
    print(f"Success rate: {success_rate}")
    print(f"Average runtime: {np.mean(runtimes)} seconds")
    print(f"Average number of Grover iterations: {np.mean(grover_iterations)}")
    print(f"Average number of Grover measurements: {np.mean(grover_measurements)}")
    print(f"Average proportion of marked elements found: {np.mean(proportion_of_marked_elements_found)}")

    plot_runtimes(runtimes, RUNTIME_BINS, "Algorithm 3")
    plot_grover_iterations(grover_iterations, GROVER_ITERATION_BINS, "Algorithm 3")
    plot_proportion_of_marked_elements(proportion_of_marked_elements_found, "Algorithm 3")